# Half-moons recalculation with shared spectral readout

This notebook recomputes the half-moon experiment so that Gamma--Dirichlet (GD) and diffusion map (DM) use the same normalized spectral-clustering readout. Consequently, GD at `gamma=0` and DM at `alpha=0` use the same unweighted kernel graph and the same normalized Laplacian, so their ARI values coincide replicate-by-replicate.

Run all code cells from top to bottom. The cell `summary = main()` executes the full recalculation and writes CSV/LaTeX tables to `gdc_recalc/`.


In [1]:
import os
os.environ.setdefault('OMP_NUM_THREADS','1')
os.environ.setdefault('OPENBLAS_NUM_THREADS','1')
os.environ.setdefault('MKL_NUM_THREADS','1')
os.environ.setdefault('NUMEXPR_NUM_THREADS','1')
from dataclasses import dataclass
from typing import Dict, Tuple
import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix, diags
from scipy.sparse.linalg import eigsh
from sklearn.cluster import KMeans
from sklearn.datasets import make_moons
from sklearn.metrics import adjusted_rand_score
from sklearn.neighbors import NearestNeighbors
from joblib import Parallel, delayed

EPS=1e-12

@dataclass(frozen=True)
class ExperimentConfig:
    n_per_moon:int=150
    n_rep:int=12
    n_jobs:int=7
    k_nn:int=15
    n_clusters:int=2
    random_seed_offset:int=0
    output_dir:str='gdc_recalc'
    gamma_grid:Tuple[float,...]=(0.0,0.25,0.5,1.0,1.5,2.0,2.5)
    alpha_grid:Tuple[float,...]=(0.0,0.25,0.5,0.75,1.0)
    settings:Dict[str,Dict[str,float]]=None
    def __post_init__(self):
        if self.settings is None:
            object.__setattr__(self,'settings',{
                'clean': {'noise':0.00,'n_clutter':0},
                'jitter': {'noise':0.12,'n_clutter':0},
                'clutter': {'noise':0.08,'n_clutter':100},
            })

In [2]:
def make_half_moons(n_per_moon=150, noise=0.0, n_clutter=0, seed=0, clutter_pad=0.5):
    rng=np.random.default_rng(seed)
    X,y=make_moons(n_samples=2*n_per_moon, noise=noise, random_state=int(rng.integers(1<<31)))
    if n_clutter>0:
        lower=X.min(axis=0)-clutter_pad
        upper=X.max(axis=0)+clutter_pad
        clutter=rng.uniform(lower,upper,size=(int(n_clutter),2))
        X=np.vstack([X,clutter])
        y=np.concatenate([y, np.full(int(n_clutter), -1, dtype=int)])
    return X,y

def gaussian_knn_kernel(X, k=15, bandwidth=None):
    n=X.shape[0]
    nn=NearestNeighbors(n_neighbors=k+1).fit(X)
    distances, indices=nn.kneighbors(X)
    if bandwidth is None:
        bandwidth=float(np.median(distances[:,1:]))
    rows=np.repeat(np.arange(n),k)
    cols=indices[:,1:].ravel()
    weights=np.exp(-(distances[:,1:].ravel()**2)/(2.0*bandwidth**2))
    K=csr_matrix((weights,(rows,cols)),shape=(n,n))
    K=K.maximum(K.T)
    return K, bandwidth

def normalized_spectral_clustering(affinity, n_clusters=2, seed=0):
    degrees=np.asarray(affinity.sum(axis=1)).ravel()
    D_inv_sqrt=diags(1.0/np.sqrt(degrees+EPS))
    S=D_inv_sqrt@affinity@D_inv_sqrt
    S=0.5*(S+S.T)
    n_eigs=min(n_clusters+1, affinity.shape[0]-1)
    v0=np.ones(affinity.shape[0])
    vals, vecs=eigsh(S, k=n_eigs, which='LA', tol=1e-7, maxiter=5000, v0=v0)
    order=np.argsort(vals)[::-1]
    vals=vals[order]; vecs=vecs[:,order]
    lap=1.0-vals
    U=vecs[:,:n_clusters]
    U=U/(np.linalg.norm(U, axis=1, keepdims=True)+EPS)
    labels=KMeans(n_clusters=n_clusters, n_init=20, random_state=seed, algorithm='lloyd').fit_predict(U)
    gap=float(lap[n_clusters]-lap[n_clusters-1]) if len(lap)>n_clusters else np.nan
    return labels, lap, gap

def gamma_dirichlet_clustering(K, gamma, n_clusters=2, seed=0):
    degree=np.asarray(K.sum(axis=1)).ravel()
    q=(degree+EPS)/np.sum(degree+EPS)
    Q=diags(q**gamma)
    C=Q@K@Q
    return normalized_spectral_clustering(C, n_clusters, seed)

def diffusion_map_old(K, alpha, n_clusters=2, diffusion_time=1.0, seed=0):
    p=np.asarray(K.sum(axis=1)).ravel()+EPS
    P=diags(p**(-alpha))
    K_alpha=P@K@P
    d_alpha=np.asarray(K_alpha.sum(axis=1)).ravel()
    D_inv_sqrt=diags(1.0/np.sqrt(d_alpha+EPS))
    S=D_inv_sqrt@K_alpha@D_inv_sqrt
    S=0.5*(S+S.T)
    n_eigs=min(n_clusters+1, K.shape[0]-1)
    vals,vecs=eigsh(S,k=n_eigs,which='LA',tol=1e-7,maxiter=5000,v0=np.ones(K.shape[0]))
    order=np.argsort(vals)[::-1]
    vals=vals[order]; vecs=vecs[:,order]
    lap=1.0-vals
    psi=D_inv_sqrt@vecs
    coords=psi[:,1:n_clusters]*(np.abs(vals[1:n_clusters])**diffusion_time)
    labels=KMeans(n_clusters=n_clusters,n_init=20,random_state=seed,algorithm='lloyd').fit_predict(coords)
    gap=float(lap[n_clusters]-lap[n_clusters-1]) if len(lap)>n_clusters else np.nan
    return labels, lap, gap

def diffusion_map_shared_readout(K, alpha, n_clusters=2, seed=0):
    p=np.asarray(K.sum(axis=1)).ravel()+EPS
    P=diags(p**(-alpha))
    K_alpha=P@K@P
    return normalized_spectral_clustering(K_alpha, n_clusters, seed)

def ari_nonclutter(y, labels):
    mask=y>=0
    return float(adjusted_rand_score(y[mask], labels[mask]))

def run_one(seed, setting, cfg):
    X,y=make_half_moons(cfg.n_per_moon,float(setting['noise']),int(setting['n_clutter']),seed+cfg.random_seed_offset)
    K,_=gaussian_knn_kernel(X,k=cfg.k_nn)
    gamma_ari=[]; gamma_gap=[]
    for gamma in cfg.gamma_grid:
        lab,_,gap=gamma_dirichlet_clustering(K,gamma,cfg.n_clusters,seed)
        gamma_ari.append(ari_nonclutter(y,lab)); gamma_gap.append(gap)
    dm_old_ari=[]; dm_new_ari=[]; dm_new_gap=[]
    for alpha in cfg.alpha_grid:
        lab,_,_=diffusion_map_old(K,alpha,cfg.n_clusters,1.0,seed)
        dm_old_ari.append(ari_nonclutter(y,lab))
        lab,_,gap=diffusion_map_shared_readout(K,alpha,cfg.n_clusters,seed)
        dm_new_ari.append(ari_nonclutter(y,lab)); dm_new_gap.append(gap)
    # sanity: gamma=0 and corrected DM alpha=0 should agree
    same0 = abs(gamma_ari[0]-dm_new_ari[0]) < 1e-12
    return np.array(gamma_ari), np.array(gamma_gap), np.array(dm_old_ari), np.array(dm_new_ari), np.array(dm_new_gap), same0

def fmt(m,s): return f'{m:.3f} ({s:.3f})'

In [3]:
# ============================================================
# Main routine and table writers
# ============================================================

def _format_mean_sd(mean, sd, digits=3):
    return f"{mean:.{digits}f} ({sd:.{digits}f})"


def _make_wide_table(summary, method, parameter_name):
    sub = summary[summary["method"] == method].copy()
    sub["entry"] = [
        _format_mean_sd(m, s)
        for m, s in zip(sub["mean_ari"], sub["sd_ari"])
    ]
    table = sub.pivot(index="setting", columns="parameter", values="entry")
    table.columns = [f"{parameter_name}={v:g}" for v in table.columns]
    return table.reset_index()


def _make_compact_table(summary):
    setting_order = ["clean", "jitter", "clutter"]
    selected_gammas = [0.0, 1.5, 2.5]
    selected_alphas = [0.0, 1.0]
    rows = []
    for setting in setting_order:
        row = {"setting": setting}
        for gamma in selected_gammas:
            sub = summary[
                (summary["setting"] == setting)
                & (summary["method"] == "Gamma-Dirichlet")
                & (summary["parameter"] == gamma)
            ].iloc[0]
            row[f"GD $\\gamma={gamma:g}$"] = _format_mean_sd(sub["mean_ari"], sub["sd_ari"])
        for alpha in selected_alphas:
            sub = summary[
                (summary["setting"] == setting)
                & (summary["method"] == "Diffusion map shared readout")
                & (summary["parameter"] == alpha)
            ].iloc[0]
            row[f"DM $\\alpha={alpha:g}$"] = _format_mean_sd(sub["mean_ari"], sub["sd_ari"])
        rows.append(row)
    return pd.DataFrame(rows)


def main(config=None):
    """Run the half-moons recalculation and save summary tables.

    The corrected diffusion-map baseline uses the same normalized spectral-
    clustering readout as Gamma--Dirichlet. Therefore GD at gamma=0 and DM at
    alpha=0 are checked to coincide replicate-by-replicate.
    """
    if config is None:
        config = ExperimentConfig()
    os.makedirs(config.output_dir, exist_ok=True)

    rows = []
    all_same0 = []

    for setting_name, setting in config.settings.items():
        print(f"Running setting: {setting_name}")
        outputs = Parallel(n_jobs=config.n_jobs)(
            delayed(run_one)(seed, setting, config)
            for seed in range(config.n_rep)
        )

        gamma_ari = np.vstack([out[0] for out in outputs])
        gamma_gap = np.vstack([out[1] for out in outputs])
        dm_old_ari = np.vstack([out[2] for out in outputs])
        dm_shared_ari = np.vstack([out[3] for out in outputs])
        dm_shared_gap = np.vstack([out[4] for out in outputs])
        same0 = np.array([out[5] for out in outputs], dtype=bool)
        all_same0.extend(same0.tolist())

        for j, gamma in enumerate(config.gamma_grid):
            rows.append({
                "setting": setting_name,
                "method": "Gamma-Dirichlet",
                "parameter_name": "gamma",
                "parameter": gamma,
                "mean_ari": gamma_ari[:, j].mean(),
                "sd_ari": gamma_ari[:, j].std(),
                "mean_eigengap": gamma_gap[:, j].mean(),
                "sd_eigengap": gamma_gap[:, j].std(),
            })

        for j, alpha in enumerate(config.alpha_grid):
            rows.append({
                "setting": setting_name,
                "method": "Diffusion map shared readout",
                "parameter_name": "alpha",
                "parameter": alpha,
                "mean_ari": dm_shared_ari[:, j].mean(),
                "sd_ari": dm_shared_ari[:, j].std(),
                "mean_eigengap": dm_shared_gap[:, j].mean(),
                "sd_eigengap": dm_shared_gap[:, j].std(),
            })
            rows.append({
                "setting": setting_name,
                "method": "Diffusion map old readout",
                "parameter_name": "alpha",
                "parameter": alpha,
                "mean_ari": dm_old_ari[:, j].mean(),
                "sd_ari": dm_old_ari[:, j].std(),
                "mean_eigengap": np.nan,
                "sd_eigengap": np.nan,
            })

    summary = pd.DataFrame(rows)

    print("\nSanity check: GD gamma=0 and DM alpha=0 coincide replicate-by-replicate:",
          bool(np.all(all_same0)))

    # Save long summary.
    long_path = os.path.join(config.output_dir, "half_moons_recalc_long.csv")
    summary.to_csv(long_path, index=False)

    # Save wide tables.
    gamma_table = _make_wide_table(summary, "Gamma-Dirichlet", "gamma")
    dm_shared_table = _make_wide_table(summary, "Diffusion map shared readout", "alpha")
    dm_old_table = _make_wide_table(summary, "Diffusion map old readout", "alpha")
    compact_table = _make_compact_table(summary)

    outputs = {
        "half_moons_gamma_table": gamma_table,
        "half_moons_dm_shared_readout_table": dm_shared_table,
        "half_moons_dm_old_readout_table": dm_old_table,
        "half_moons_compact_shared_readout_table": compact_table,
    }
    for name, table in outputs.items():
        csv_path = os.path.join(config.output_dir, f"{name}.csv")
        tex_path = os.path.join(config.output_dir, f"{name}.tex")
        table.to_csv(csv_path, index=False)
        table.to_latex(tex_path, index=False, escape=False)
        print(f"Saved {csv_path}")
        print(f"Saved {tex_path}")

    print(f"Saved {long_path}")
    print("\nCompact manuscript table:")
    print(compact_table.to_string(index=False))

    return summary


## Run the full recalculation

This cell writes the long-format summary and the wide CSV/LaTeX tables to `gdc_recalc/`. In Colab, change `output_dir` in `ExperimentConfig` if desired.

In [4]:
summary = main()
summary.head()

Running setting: clean
Running setting: jitter
Running setting: clutter

Sanity check: GD gamma=0 and DM alpha=0 coincide replicate-by-replicate: True
Saved gdc_recalc/half_moons_gamma_table.csv
Saved gdc_recalc/half_moons_gamma_table.tex
Saved gdc_recalc/half_moons_dm_shared_readout_table.csv
Saved gdc_recalc/half_moons_dm_shared_readout_table.tex
Saved gdc_recalc/half_moons_dm_old_readout_table.csv
Saved gdc_recalc/half_moons_dm_old_readout_table.tex
Saved gdc_recalc/half_moons_compact_shared_readout_table.csv
Saved gdc_recalc/half_moons_compact_shared_readout_table.tex
Saved gdc_recalc/half_moons_recalc_long.csv

Compact manuscript table:
setting GD $\gamma=0$ GD $\gamma=1.5$ GD $\gamma=2.5$ DM $\alpha=0$ DM $\alpha=1$
  clean 0.733 (0.378)   1.000 (0.000)   1.000 (0.000) 0.733 (0.378) 1.000 (0.000)
 jitter 0.797 (0.174)   0.968 (0.025)   0.632 (0.378) 0.797 (0.174) 0.452 (0.143)
clutter 0.631 (0.384)   0.997 (0.008)   0.607 (0.407) 0.631 (0.384) 0.076 (0.070)


,setting,method,parameter_name,parameter,mean_ari,sd_ari,mean_eigengap,sd_eigengap
0,clean,Gamma-Dirichlet,gamma,0.00,0.73255,0.378232,0.002117,0.001495
1,clean,Gamma-Dirichlet,gamma,0.25,1.00000,0.000000,0.003215,0.000002
2,clean,Gamma-Dirichlet,gamma,0.50,1.00000,0.000000,0.003251,0.000002
3,clean,Gamma-Dirichlet,gamma,1.00,1.00000,0.000000,0.003309,0.000002
4,clean,Gamma-Dirichlet,gamma,1.50,1.00000,0.000000,0.003352,0.000002


## Compact table for the manuscript

This extracts representative parameter values for the short manuscript table.

In [6]:
# ============================================================
# Compact manuscript table used in the paper
# ============================================================
# This cell uses the full summary CSV produced above and extracts the
# representative columns used in Table 1.

import os
import pandas as pd

output_dir = ExperimentConfig().output_dir
summary = pd.read_csv(os.path.join(output_dir, "half_moons_recalc_long.csv"))

setting_order = ["clean", "jitter", "clutter"]
selected_gammas = [0.0, 1.5, 2.5]
selected_alphas = [0.0, 1.0]

def entry(row):
    return f"{row['mean_ari']:.3f} ({row['sd_ari']:.3f})"

rows = []
for setting in setting_order:
    row = {"setting": setting}
    for gamma in selected_gammas:
        sub = summary[
            (summary["setting"] == setting)
            & (summary["method"] == "Gamma-Dirichlet")
            & (summary["parameter"] == gamma)
        ].iloc[0]
        row[f"GD $\\gamma={gamma:g}$"] = entry(sub)
    for alpha in selected_alphas:
        sub = summary[
            (summary["setting"] == setting)
            & (summary["method"] == "Diffusion map shared readout")
            & (summary["parameter"] == alpha)
        ].iloc[0]
        row[f"DM $\\alpha={alpha:g}$"] = entry(sub)
    rows.append(row)

compact = pd.DataFrame(rows)
compact_path = os.path.join(output_dir, "half_moons_compact_shared_readout_table.csv")
compact_tex_path = os.path.join(output_dir, "half_moons_compact_shared_readout_table.tex")
compact.to_csv(compact_path, index=False)
compact.to_latex(compact_tex_path, index=False, escape=False)

print(compact.to_string(index=False))
print("\nSaved:")
print(compact_path)
print(compact_tex_path)

setting GD $\gamma=0$ GD $\gamma=1.5$ GD $\gamma=2.5$ DM $\alpha=0$ DM $\alpha=1$
  clean 0.733 (0.378)   1.000 (0.000)   1.000 (0.000) 0.733 (0.378) 1.000 (0.000)
 jitter 0.797 (0.174)   0.968 (0.025)   0.632 (0.378) 0.797 (0.174) 0.452 (0.143)
clutter 0.631 (0.384)   0.997 (0.008)   0.607 (0.407) 0.631 (0.384) 0.076 (0.070)

Saved:
gdc_recalc/half_moons_compact_shared_readout_table.csv
gdc_recalc/half_moons_compact_shared_readout_table.tex
